# Simplified TorchGeo Semantic Segmentation

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/torchgeo_segmentation_simple.ipynb)

This notebook shows the high-level TorchGeo semantic segmentation workflow in GeoAI. It trains directly from aligned GeoTIFF imagery and masks without exporting chip files first.


## Install packages


In [ ]:
# %pip install geoai-py torchgeo matplotlib

## Import libraries


In [ ]:
import geoai

## Download sample data

This uses the same NAIP water dataset as `train_water_detection.ipynb`.


In [ ]:
train_raster_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip/naip_water_train.tif"
train_masks_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip/naip_water_masks.tif"

train_raster_path = geoai.download_file(train_raster_url)
train_masks_path = geoai.download_file(train_masks_url)

## Visualize the mask over the image


In [ ]:
geoai.view_raster(train_masks_path, nodata=0, basemap=train_raster_path)

## Train from TorchGeo samples

`train_torchgeo_segmentation_model` creates a TorchGeo segmentation dataset, samples training/validation patches, trains a small segmentation model, and returns the model, dataloaders, and training history.


In [ ]:
result = geoai.train_torchgeo_segmentation_model(
    image_path=train_raster_path,
    mask_path=train_masks_path,
    output_dir="torchgeo_water_simple",
    chip_size=256,
    train_length=64,
    val_length=16,
    batch_size=4,
    num_epochs=20,
    learning_rate=1e-3,
    num_channels=3,
    num_classes=2,
)

### About the best score

The training output reports `best_val_loss`. Because it is a loss, lower is better. A decreasing best score means validation loss is improving. For metrics such as IoU or accuracy, higher is better.


In [ ]:
result["history"]

## Plot performance metrics


In [ ]:
metrics_df = geoai.plot_performance_metrics(
    history_path=result["history_path"],
    figsize=(10, 4),
    verbose=True,
)
metrics_df

## Visualize predictions


In [ ]:
geoai.plot_torchgeo_segmentation_predictions(
    result["model"],
    result["val_loader"],
    n=4,
)

## Optional: create dataloaders for DINOv3

The same high-level dataloader helper can prepare TorchGeo dataloaders for `train_dinov3_segmentation`. The DINOv3 call is commented out because it downloads large model weights and is better run with a GPU.


In [ ]:
loaders = geoai.create_torchgeo_segmentation_dataloaders(
    train_raster_path,
    train_masks_path,
    chip_size=256,
    train_length=64,
    val_length=16,
    batch_size=4,
)

# dinov3_model = geoai.train_dinov3_segmentation(
#     train_dataloader=loaders["train"],
#     val_dataloader=loaders["val"],
#     output_dir="dinov3_torchgeo_output",
#     num_classes=2,
#     num_epochs=10,
#     monitor_metric="val_loss",
#     mode="min",
# )

## Summary

This simplified workflow mirrors the chip-based training examples, but the patches are sampled directly from georeferenced rasters at training time.
